In [0]:
# Configuration
from pyspark.sql import functions as F
from datetime import datetime


In [0]:
# Imports and configuration
UTILS_DIR = "/Workspace/Repos/ts.maira@gmail.com/techmart-catalog-pipeline/utils"

exec(open(f"{UTILS_DIR}/config.py").read())
exec(open(f"{UTILS_DIR}/states.py").read())
exec(open(f"{UTILS_DIR}/llm_utils.py").read())

PROCESSED_AT      = datetime.now().isoformat()
print("✅ Utils loaded")


In [0]:
# Read all three tables
df_taxonomy = spark.table(SILVER_TAXONOMY)
                #    .filter(F.col("judge_approved") == "True")
df_vendors  = spark.table(SILVER_VENDORS)
df_products = spark.table(SILVER_PRODUCTS)

print(f"Approved taxonomy records : {df_taxonomy.count()}")
print(f"Vendors                   : {df_vendors.count()}")
print(f"Products                  : {df_products.count()}")

In [0]:
# Join taxonomy with vendors and products
# We join on product_id to enrich each LLM result with:
# - vendor_name from silver_vendors
# - unit_price_usd and weight_kg from silver_products

df_enriched = (
    df_taxonomy
    .join(df_vendors,  on="product_id", how="left")
    .join(
        df_products.select("product_id", "unit_price_usd", "weight_kg"),
        on="product_id",
        how="left"
    )
    .select(
        F.col("product_id").cast("integer"),
        F.col("extracted_name").alias("product_name"),
        F.col("extracted_brand").alias("brand"),
        F.col("extracted_sub_category").alias("sub_category"),
        F.col("vendor_name"),
        F.col("unit_price_usd"),
        F.col("weight_kg"),
        F.col("judge_approved"),
        F.col("judge_reason"),
        F.col("prompt_version"),
        F.lit(PROCESSED_AT).alias("_processed_at")
    )
)

print(f"Enriched records: {df_enriched.count()}")
display(df_enriched)

In [0]:
# Add this after the join in Cell 3
category_map = F.create_map(
    F.lit("Televisions"),  F.lit("Entertainment"),
    F.lit("Computers"),    F.lit("Computing"),
    F.lit("Phones"),       F.lit("Mobile Devices"),
    F.lit("Smartwatches"), F.lit("Wearables"),
    F.lit("Accessories"),  F.lit("Accessories"),
    F.lit("Printers"),     F.lit("Computing"),
    F.lit("Cameras"),      F.lit("Photography"),
    F.lit("Consoles"),     F.lit("Entertainment"),
    F.lit("Hardware"),     F.lit("Computing")
)

df_enriched = df_enriched.withColumn(
    "category",
    F.coalesce(category_map[F.col("sub_category")], F.lit("Other"))
)

In [0]:
# Write enriched taxonomy to Delta
(df_enriched.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    # .option("overwriteSchema", "true")
    .saveAsTable(TAXONOMY_ENRICHED))

print(f"✅ Written: {TAXONOMY_ENRICHED}")

In [0]:
# Validate
print("=== VALIDATION ===")
print(f"Total enriched records: {spark.table(TAXONOMY_ENRICHED).count()}")
display(spark.table(TAXONOMY_ENRICHED))